In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Controls — immediately after Drive mount. Safe for a brand-new Runtime → Run all.
DRIVE_ROOT='/content/drive/MyDrive/OpenPlaque'
OUTPUT_ROOT=DRIVE_ROOT + '/LCX_Curved_Template_Reacquisition_v1_cacheonly_fixed'
BRANCH='lcx-curved-template-reacquisition-from-main'


# OpenPlaque — LCX curved-template reacquisition — cache-only fixed

This notebook fixes the prior runtime failure caused by the missing `OpenPlaque/Full_DICOM.zip`. It uses the already cached curved-series CT NIfTIs and nnU-Net masks in `UCLA_Plaque_Context_Verification`, via `lcx_curved_template_reacquisition_v2.py`.

Scientific interpretation is unchanged: the historical LCX-labeled mask is an **unlabeled coronary template**. A passing match only nominates a source-space path for later anatomical/topological QC; it does not establish LCX identity or source-space plaque volume.


In [ ]:
import shutil, subprocess, sys
REPO='/content/OpenPlaque'
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git',REPO], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pytest','scikit-image','pydicom','SimpleITK'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO], check=True)
print('COMMIT:')
subprocess.run(['git','-C',REPO,'rev-parse','HEAD'], check=True)
print('SEPARATE-PROCESS IMPORT CHECK:')
subprocess.run([sys.executable,'-c',"import openplaque; import openplaque.lcx_curved_template_reacquisition_v2 as m; print(openplaque.__file__); print(m.ALGORITHM)"], check=True)


In [ ]:
# Syntax and regression tests using normal installed-package imports.
subprocess.run([sys.executable,'-m','py_compile',
                REPO + '/src/openplaque/lcx_curved_template_reacquisition.py',
                REPO + '/src/openplaque/lcx_curved_template_reacquisition_v2.py'], check=True)
subprocess.run([sys.executable,'-m','pytest','-q',
                REPO + '/tests/test_lcx_curved_template_reacquisition.py',
                REPO + '/tests/test_lcx_curved_template_reacquisition_v2.py'], check=True)


In [ ]:
# Preflight every required cached input before running the workflow.
from pathlib import Path
root=Path(DRIVE_ROOT)
required=[
 root/'Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
 root/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
 root/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries_LEGACY/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/total/aorta.nii.gz',
 root/'UCLA_Plaque_Context_Verification/RCA_input/RCA_0000.nii.gz',
 root/'UCLA_Plaque_Context_Verification/LCX_input/LCX_0000.nii.gz',
 root/'UCLA_Plaque_Context_Verification/nnunet_masks/RCA.nii.gz',
 root/'UCLA_Plaque_Context_Verification/nnunet_masks/LCX.nii.gz',
]
missing=[str(p) for p in required if not p.exists()]
print(f'INPUT PREFLIGHT: {len(required)-len(missing)}/{len(required)} present')
for p in required:
    print(('OK   ' if p.exists() else 'MISS '), p)
if missing:
    raise FileNotFoundError('Missing required cached inputs:\n'+'\n'.join(missing))


In [ ]:
# Run the actual scientific workflow in a fresh Python subprocess.
# No PYTHONPATH/sys.path manipulation is used.
from pathlib import Path
import subprocess, sys
runner=Path('/content/run_lcx_cacheonly_fixed.py')
runner.write_text(f'''
from openplaque.lcx_curved_template_reacquisition_v2 import synthetic_lcx_template_v2_self_test, run
print('SELF TEST:', synthetic_lcx_template_v2_self_test(), flush=True)
result=run({DRIVE_ROOT!r}, {OUTPUT_ROOT!r})
print('STATUS:', result['summary']['status'], flush=True)
print('REPORT:', result['report'], flush=True)
print('ZIP:', result['zip'], flush=True)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LCX_CURVED_TEMPLATE_REACQUISITION_REPORT_BACK.zip', flush=True)
''', encoding='utf-8')
proc=subprocess.run([sys.executable,str(runner)], text=True, capture_output=True)
print(proc.stdout)
if proc.stderr:
    print('--- WORKFLOW STDERR ---')
    print(proc.stderr)
if proc.returncode != 0:
    state=Path(OUTPUT_ROOT)/'run_state.json'
    if state.exists():
        print('--- run_state.json ---')
        print(state.read_text())
    raise RuntimeError(f'LCX cache-only workflow failed with exit code {proc.returncode}; full child traceback is printed above.')


In [ ]:
# Display outputs after success.
from pathlib import Path
from IPython.display import display, Image
out=Path(OUTPUT_ROOT)
for name in ['01_curved_template_fingerprints.png','02_candidate_match_scores.png','03_top_candidate_source_orthogonal_qc.png']:
    p=out/name
    if p.exists():
        display(Image(filename=str(p)))
report=out/'OPENPLAQUE_LCX_CURVED_TEMPLATE_REACQUISITION_REPORT.html'
zip_path=out/'OPENPLAQUE_LCX_CURVED_TEMPLATE_REACQUISITION_REPORT_BACK.zip'
print('REPORT:', report)
print('ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LCX_CURVED_TEMPLATE_REACQUISITION_REPORT_BACK.zip')
